# ДЗ 4. Операции, axis и broadcasting на данных «Дала Маркет»
Срок: **до понедельника 16.11, 19:00**.
Сдача: папка `hw04` репозитория `dsa-homework` (`hw04.ipynb`): `git add hw04` → `git commit` → `git push`, ссылку прикрепите в LMS.

Каждый ответ: **число + одна фраза для менеджера**. Циклы `for` по заказам не используем. Небольшой цикл по городам или категориям при сборке таблицы допустим.

- **[База]** обязательно для всех · **[Стандарт]** сколько получится · **[Челлендж]** для тех, кому легко.

## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
import numpy as np
# 1
a = np.array([2, 4, 6]); print(a / 2, a // 4)
# 2
m = np.ones((3, 4)); print(m.sum(axis=1))
# 3
print(np.arange(3) + np.arange(3).reshape(3, 1))
# 4
x = np.array([1, 4, 9]); print(np.sqrt(x).sum())
# 5
print((np.arange(1, 5) ** 2).mean())
```

Ваши ответы и объяснения:

1. 
2. 
3. 
4. 
5. 

In [2]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

orders.csv на месте: True


In [3]:
import csv
import numpy as np

CITY = {"алматы": "Алматы", "almaty": "Алматы", "астана": "Астана", "astana": "Астана",
        "шымкент": "Шымкент", "shymkent": "Шымкент", "караганда": "Караганда",
        "актобе": "Актобе", "атырау": "Атырау"}

with open("orders.csv", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

amount = np.array([float(r["amount"].replace(" ", "")) for r in rows])
paid   = np.array([r["status"] == "оплачен" for r in rows])
days   = np.array([float(r["delivery_days"]) if r["delivery_days"] else np.nan for r in rows])
city   = np.array([CITY.get(r["city"].strip().lower(), r["city"].strip()) for r in rows])
cust   = np.array([r["customer_id"] for r in rows])
month  = np.array([r["order_date"][:7] for r in rows])      # "2026-09" или "2026-10"
cat    = np.array([r["category"] for r in rows])
oid    = np.array([r["order_id"] for r in rows])

print(amount.shape, np.unique(month))

(5463,) ['2026-09' '2026-10']


In [4]:
# Чистые данные: без дублей (оставляем первое вхождение order_id), без TEST, только оплаченные
_, first = np.unique(oid, return_index=True)
keep = np.zeros(len(oid), dtype=bool)
keep[first] = True
clean = keep & (cust != "TEST") & paid
print("строк всего:", len(oid), "| в чистых данных:", clean.sum())

строк всего: 5463 | в чистых данных: 4739


## [База] 1. Выручка по чистым данным
Сколько строк осталось в `clean`? Выручка за сентябрь и октябрь и изменение в %. Совпадает ли с ДЗ 2?

In [14]:
# 1
print(f'Строк в `clean`:', clean.sum())

Строк в `clean`: 4739


In [15]:
sep = amount[clean & (month == "2026-09")].sum()
oct = amount[clean & (month == "2026-10")].sum()

print(f'Выручка за сентябрь:', sep)
print(f'Выручка за октябрь:', oct)
print(f'Изменение: {(oct - sep) / sep * 100:.2f}%')

Выручка за сентябрь: 35243410.0
Выручка за октябрь: 34419520.0
Изменение: -2.34%


Из ДЗ 2: 5463 строк, выручка за сентябрь: 57489850.0, за октябрь: 44419510.0.

Совпадает ли с ДЗ 2? Нет, потому что рассчет выручек через `clean` обозначает, что мы считаем оплаченные заказы, но исключаем из подсчетов клиента TEST и дубликаты строк с одним id заказа.

## [База] 2. Таблица «категория × месяц»
Соберите таблицу 5 × 2 по образцу таблицы городов с урока (категории: `np.unique(cat)`).
Какая категория приносит больше всего выручки за два месяца (`axis`)? Какая упала сильнее всех в %?

In [ ]:
# 2

In [65]:
months = np.array(["2026-09", "2026-10"])
categories = np.unique(cat)

cat_table = np.array([[amount[clean & (cat == c) & (month == m)].sum() for m in months]
                      for c in categories])

print(f"{'Категория':<12} {'Сентябрь':>12} {'Октябрь':>12} {'Всего':>12} {'Разница':>8}")
for c, r in zip(categories, cat_table):
    print(f"{c:<12} {r[0]:>12,.0f} {r[1]:>12,.0f} {r.sum():>12,.0f} {((r[1] - r[0]) / r[0] * 100):>7,.1f}%")
print(cat_table.shape)

Категория        Сентябрь      Октябрь        Всего  Разница
Дом             4,997,730    5,125,790   10,123,520     2.6%
Книги           1,065,300    1,140,910    2,206,210     7.1%
Красота         2,610,950    2,678,990    5,289,940     2.6%
Одежда          9,167,100    8,505,730   17,672,830    -7.2%
Электроника    17,402,330   16,968,100   34,370,430    -2.5%
(5, 2)


In [50]:
total = cat_table.sum(axis=1)
print(categories[total.argmax()], total.max())

Электроника 34370430.0


In [51]:
change = (cat_table[:, 1] - cat_table[:, 0]) / cat_table[:, 0] * 100
print(categories[change.argmin()], change.min())

Одежда -7.214604400519248


Больше всего выручки за 2 месяца приносит электроника (34 370 430). Сильнее всех упала категория Одежда (-7.2%).

## [База] 3. Средний чек по категориям в октябре
Соберите вторую таблицу: **число** заказов по категориям и месяцам (сумма маски, а не сумма amount).
Средний чек = выручка / число заказов, одной операцией над двумя таблицами. У какой категории он самый высокий?

In [ ]:
# 3

In [72]:
count_table = np.array([[(clean & (cat == c) & (month == m)).sum() for m in months]
                      for c in categories])

print(f"{'Категория':<12} {'Сентябрь':>6} {'Октябрь':>6}")
for c, r in zip(categories, count_table):
    print(f"{c:<12} {r[0]:>6,.0f} {r[1]:>6,.0f}")

Категория    Сентябрь Октябрь
Дом             476    476
Книги           265    299
Красота         422    435
Одежда          678    652
Электроника     531    505


In [76]:
avg = cat_table / count_table

print(f"{'Категория':<12} {'Сентябрь':>6} {'Октябрь':>6}")
for c, r in zip(categories, avg):
    print(f"{c:<12} {r[0]:>7,.0f} {r[1]:>7,.0f}")

Категория    Сентябрь Октябрь
Дом           10,499  10,768
Книги          4,020   3,816
Красота        6,187   6,159
Одежда        13,521  13,046
Электроника   32,773  33,600


In [80]:
print(categories[avg[:, 1].argmax()], avg[:, 1].max())

Электроника 33600.19801980198


Самый высокий средний чек в октябре - в категории Электроника (33 600)

## [Стандарт] 4. Доля городов в выручке месяца
Используйте `table` городов с урока. Доля каждого города в выручке своего месяца: `table / table.sum(axis=0)`.
Проверьте, что каждый столбец в сумме даёт 1. Как изменилась доля Шымкента?

In [ ]:
# 4

In [83]:
cities = np.unique(city)
months = np.array(["2026-09", "2026-10"])

# Таблица 6 × 2: строка = город, столбец = месяц
table = np.array([[amount[clean & (city == c) & (month == m)].sum() for m in months]
                  for c in cities])
for c, r in zip(cities, table):
    print(f"{c:<10} {r[0]:>12,.0f} {r[1]:>12,.0f}")

Актобе        2,576,960    2,540,420
Алматы       14,501,070   14,643,020
Астана        9,034,220    8,850,530
Атырау          923,990      969,860
Караганда     3,648,520    3,371,680
Шымкент       4,558,650    4,044,010


In [116]:
share = table / table.sum(axis=0)
print('Доля каждого города:')

for c, r in zip(cities, share):
    print(f"{c:<10} {r[0]:>12,.4f} {r[1]:>12,.4f}")

Доля каждого города:
Актобе           0.0731       0.0738
Алматы           0.4115       0.4254
Астана           0.2563       0.2571
Атырау           0.0262       0.0282
Караганда        0.1035       0.0980
Шымкент          0.1293       0.1175


In [117]:
print(share.sum(axis=0))

[1. 1.]


Сложили значения долей сверху вниз по строкам и убедились что каждый столбец дает в сумме 1.

In [106]:
shymkent = share[cities == "Шымкент"]
print(shymkent * 100)

[[12.93475858 11.74917605]]


In [118]:
print((shymkent[0, 1] - shymkent[0, 0]) * 100)

-1.1855825342961728


Доля Шымкента уменьшилась с 12.93% в сентябре до 11.75% в октябре, то есть примерно на 1.19 процентных пункта.

In [120]:
print(((shymkent[0, 1] - shymkent[0, 0])) / shymkent[0, 0] * 100)

-9.165865189264165


Доля Шымкента уменьшилась с 12.93% в сентябре до 11.75% в октябре, то есть примерно на 1.19 процентных пункта, или на 9.17%.

## [Стандарт] 5. Кто дал падение
Вклад каждого города в общее падение выручки: изменение города / общее изменение. Сколько процентов падения даёт Шымкент?
Напишите менеджеру одну фразу: где искать причину.

In [ ]:
# 5

## [Стандарт] 6. Исправьте код от AI
AI написал код «доля сентября и октября внутри каждого города». Код падает. Объясните, почему, и исправьте.
```python
share = table / table.sum(axis=1)
```

In [95]:
# 6
print(table)
print(table.shape)

[[ 2576960.  2540420.]
 [14501070. 14643020.]
 [ 9034220.  8850530.]
 [  923990.   969860.]
 [ 3648520.  3371680.]
 [ 4558650.  4044010.]]
(6, 2)


In [96]:
print(table.sum(axis=1))
print(table.sum(axis=1).shape)

[ 5117380. 29144090. 17884750.  1893850.  7020200.  8602660.]
(6,)


Код падает потому что выходит ошибка `ValueError: operands could not be broadcast together with shapes (6,2) (6,) `. То есть мы можем выполнять действия с массивами одной формы, а они у нас не одной формы. Мы не можем превратить (6, ) в (6, 2). Но можем сохранить результаты одномерного массива (6, ) в двумерном массиве (6, 1) через keepdims=True. А затем (6, 1) растянуть в (6, 2). Это возможно потому что чтобы сработал broadcasting, размеры должны совпадать или один из них должен быть равен 1.

Исправленный код ниже.

In [99]:
share = table / table.sum(axis=1, keepdims=True)
print(share)

[[0.50357019 0.49642981]
 [0.49756469 0.50243531]
 [0.50513538 0.49486462]
 [0.48788975 0.51211025]
 [0.51971739 0.48028261]
 [0.52991168 0.47008832]]


## [Челлендж] 7. Стандартизация суммы заказа
Для чистых оплаченных заказов посчитайте z-оценку: `z = (x - x.mean()) / x.std()`.
Сколько заказов с |z| > 3 и какая это доля? Проверьте, что у `z` среднее ≈ 0 и std = 1.

In [ ]:
# 7

## [Челлендж] 8. Масштабирование признаков (заготовка для ML)
Соберите матрицу `X` из двух столбцов: `amount` и `days` для чистых заказов без пропусков в `days` (`np.column_stack`).
Приведите каждый столбец к диапазону [0, 1] одной строкой: `(X - X.min(axis=0)) / (X.max(axis=0) - X.min(axis=0))`.
Проверьте min и max каждого столбца. Объясните своими словами, зачем модели нужны признаки в одном масштабе.

In [ ]:
# 8

Комментарии: Модели нужны признаки в одном масштабе для того, чтобы привести признаки к одной "метрике" в диапазоне от 0 до 1. Таким образом, степень влияния условного признака X1, значение которого 1000, будет адекватно оценено по сравнению с признаком X2 (X2 = 50), и X1 не будет иметь большее влияние в модели только потому что его значение больше, чем X2.